# Time analysis

Cost of every method, from the pipeline's own timers (`runtime_profile.parquet`,
written by `02_modelling.py`). Same run as `03_results.ipynb`.

Curve training runs in a worker pool, so its seconds are **summed worker CPU** —
comparable between methods, larger than the run's wall clock. Everything else is
single-process wall clock.

## 0 · Setup

In [1]:
# ── Shared config ────────────────────────────────────────────────────────────
import json, re, warnings
import numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display
warnings.filterwarnings('ignore')

RESULTS_ROOT = Path('..') / 'results'

EXPERIMENT = 1          # same run as 03_results.ipynb
SPLIT      = 'TEST'     # split whose inference cost is reported
SAVE_LATEX = True

# Approach key -> the label the curve evaluation logged it under -> the paper name.
# The paper names are I_METHOD_RENAME of 03_results.ipynb, so a method is called the
# same thing in the accuracy table and here.
APPROACH_EVAL_LABEL = {
    'ml_step_dtw_smooth':     'Step DTW smooth + ML + Ext.',
    'median_activity_sensor': 'Median per Activity & Sensor',
    'ml_external':            'ML + Ext. Factors',
    'ml_only':                'ML only (no DTW)',
    'baseline':               'Baseline',
    'seq2seq_external':       'DTW + Seq2Seq + Ext. Factors',
    'seq2seq_iom':            'Seq2Seq IOM (DTW-selected)',
}
METHOD_RENAME = {
    'Step DTW smooth + ML + Ext.':  'ML Step DTW (proposed)',
    'Step DTW + ML + Ext.':         'Step DTW (step gains)',
    'ML + Ext. Factors':            'ML DTW (no steps)',
    'ML DTW':                       'Step DTW w/o segments and ext.',
    'ML only (no DTW)':             'ML (no DTW)',
    'Median per Activity & Sensor': 'Median per Activity & Sensor',
    'Baseline':                     'Sensor median',
    'DTW + Seq2Seq + Ext. Factors': 'Seq2Seq DTW (aligned)',
    'Seq2Seq IOM (DTW-selected)':   'Seq2Seq (DTW-scored)',
    'Seq2Seq only (no DTW)':        'Seq2Seq (unaligned)',
}
# Row order of table 1 = the realism ranking of the paper's Evaluation 2 table, so
# the two can be read side by side. Anything else lands at the end.
METHOD_ORDER = ['ML Step DTW (proposed)', 'Median per Activity & Sensor',
                'ML DTW (no steps)', 'ML (no DTW)', 'Sensor median',
                'Seq2Seq DTW (aligned)', 'Seq2Seq (DTW-scored)']

# Stages whose code has since been removed from 02_modelling.py: timed in this run,
# absent from a rerun. Excluded from the tables.
RETIRED_STAGES = ['energy_distribution_eval']


def _tex_esc(s):
    # The em dash is spelled '--': the stage names carry one and the paper's .tex
    # files are compiled without relying on UTF-8 input.
    return (str(s).replace('\\', r'\textbackslash ').replace('&', r'\&')
                  .replace('_', r'\_').replace('%', r'\%').replace('#', r'\#')
                  .replace('—', '--'))


def _latest_run(experiment):
    """Newest run dir for this experiment, by its TIMESTAMP (as in 03_results)."""
    pat  = re.compile(rf'^experiment_{experiment}_(\d{{8}}_\d{{6}})$')
    runs = sorted((m.group(1), d) for d in RESULTS_ROOT.iterdir()
                  if d.is_dir() and (m := pat.match(d.name)))
    assert runs, f'No runs for experiment {experiment}'
    return runs[-1][1]


def _time_variant(mode):
    """'petri_net_<net>[_ml_plus_*]' -> which activity-duration predictor it used."""
    r = str(mode)
    if r.endswith('_ml_plus_global'):  return 'ml_global'
    if r.endswith('_ml_plus_per_act'): return 'ml_local'
    return 'baseline'


In [2]:
# ── Load the timers, the curve population and the case population ───────────
run  = _latest_run(EXPERIMENT)
rt   = pd.read_parquet(run / 'runtime_profile.parquet')
info = json.loads((run / 'info.json').read_text())
N_PROCESSES = len(info['processes'])

# Inference cost is per unit of predicted work, so the denominators come from the
# evaluation outputs: curves per (approach, split), cases per (mode, split).
curves = (pd.read_parquet(run / 'curve_eval_results.parquet')
            .groupby(['Approach', 'Split']).size())
cases  = (pd.read_parquet(run / 'process_eval_per_case.parquet')
            .groupby(['mode', 'split']).size())

_w = rt['parallel_workers'].dropna()
POOL_WORKERS = int(_w.max()) if len(_w) else None
_wall = float(rt.loc[rt['stage'] == 'pipeline_total', 'seconds'].sum())

print(f'Run: {run.name} | processes: {N_PROCESSES} | approaches: '
      f'{len(info["approaches"])} | timed rows: {len(rt)}')
print(f'Pool workers: {POOL_WORKERS} | whole-run wall clock: {_wall/3600:.2f} h')

_ret = [s for s in RETIRED_STAGES if s in set(rt['stage'])]
if _ret:
    print('WARNING: ' + ', '.join(_ret) + ' — timed here, code since removed from '
          'the pipeline. Excluded.')

Run: experiment_1_20260802_085901 | processes: 6 | approaches: 7 | timed rows: 705
Pool workers: 16 | whole-run wall clock: 13.31 h


---
## 1 · Curve generation

A curve model is one predictor per (sensor, activity, object), per process. The
sensor-median baseline fits one per *sensor*, hence its smaller count — the
per-model and per-curve columns are the comparable ones.

In [3]:
# ── T1: training cost per curve model, inference cost per curve ─────────────
_tr = rt[rt['stage'] == 'curve_training'].copy()
_tr['approach'] = _tr['detail'].astype(str).str.split(':').str[-1]
_tr['pool']     = _tr['detail'].astype(str).str.split(':').str[0]

# '_curve_extraction*' is shared preprocessing every method pays, not a method.
_shared = _tr[_tr['approach'].str.startswith('_curve_extraction')]
_tr     = _tr[~_tr['approach'].str.startswith('_curve_extraction')]

t1 = (_tr.groupby('approach')
         .agg(train_s=('seconds', 'sum'),
              models=('n_items', 'sum'),
              pool=('pool', 'first')))
t1['train_per_model_s'] = t1['train_s'] / t1['models'].replace(0, np.nan)

_ev = rt[(rt['stage'] == 'curve_only_eval') & (rt['split'] == SPLIT)].copy()
_ev['approach'] = _ev['detail'].map({v: k for k, v in APPROACH_EVAL_LABEL.items()})
_ev['curves']   = _ev['detail'].map(lambda d: curves.get((d, SPLIT), np.nan))

# Every label the evaluation loop knows gets a timed block, also the ones this run
# did not train — those cost ~0 s. Only an unmapped label with seconds on the clock
# is a mapping gap.
_un  = _ev[_ev['approach'].isna()]
_gap = sorted(set(_un.loc[_un['seconds'] >= 0.05, 'detail']))
print(f'Not trained in this run, 0 s logged: '
      f'{len(set(_un.loc[_un["seconds"] < 0.05, "detail"]))}')
if _gap:
    print('WARNING: timed but unmapped, missing from the table: ' + ', '.join(_gap))

_inf = (_ev.dropna(subset=['approach']).groupby('approach')
           .agg(infer_s=('seconds', 'sum'), curves=('curves', 'sum')))
_inf['infer_per_curve_ms'] = 1000 * _inf['infer_s'] / _inf['curves'].replace(0, np.nan)

t1 = t1.join(_inf)
# Trained models but zero inference seconds = no curves for this split. Missing,
# not zero.
t1.loc[t1['curves'].fillna(0) == 0, ['infer_s', 'infer_per_curve_ms']] = np.nan

t1['Method'] = [METHOD_RENAME.get(APPROACH_EVAL_LABEL.get(a, a),
                                  APPROACH_EVAL_LABEL.get(a, a)) for a in t1.index]
t1 = t1.set_index('Method')
t1 = t1.reindex([m for m in METHOD_ORDER if m in t1.index]
                + [m for m in t1.index if m not in METHOD_ORDER])

T1 = pd.DataFrame({
    'Curve models':             t1['models'].astype('Int64'),
    'Training total (min)':     t1['train_s'] / 60,
    'Training per model (s)':   t1['train_per_model_s'],
    f'Curves ({SPLIT})':        t1['curves'],
    'Inference total (s)':      t1['infer_s'],
    'Inference per curve (ms)': t1['infer_per_curve_ms'],
})
display(T1.round(2))

_pool = rt[rt['stage'] == 'curve_training_pool'].groupby('detail')['seconds'].sum() / 60
print(f'Shared curve extraction, paid once by every method: '
      f'{_shared["seconds"].sum()/60:.1f} min / '
      f'{int(_shared["n_items"].sum()):,} extractions')
print('Pool wall clock: ' + ' | '.join(f'{k}: {v:.0f} min' for k, v in _pool.items())
      + f' — vs {t1["train_s"].sum()/60:.0f} min summed CPU over {POOL_WORKERS} workers')

Not trained in this run, 0 s logged: 6


,Curve models,Training total (min),Training per model (s),Curves (TEST),Inference total (s),Inference per curve (ms)
Method,,,,,,
ML Step DTW (proposed),390,246.65,37.95,19345.0,436.99,22.59
Median per Activity & Sensor,390,0.03,0.00,19345.0,128.40,6.64
ML DTW (no steps),390,870.83,133.97,19345.0,403.89,20.88
ML (no DTW),390,869.53,133.77,19345.0,322.03,16.65
Sensor median,79,0.65,0.49,19403.0,132.41,6.82
Seq2Seq DTW (aligned),390,164.83,25.36,19345.0,803.53,41.54
Seq2Seq (DTW-scored),390,678.74,104.42,19345.0,702.10,36.29


Shared curve extraction, paid once by every method: 6.4 min / 1,948 extractions
Pool wall clock: seq2seq: 72 min | sklearn: 179 min — vs 2831 min summed CPU over 16 workers


---
## 2 · Process-model stages

Fitting is per process, generation per generated case. Case generation is the
simulator alone; complete profile adds the curve models over a whole case and the
comparison against its real counterpart.

`Total` is what this run actually spent on the stage, over `N` units — for the
generation stages that is every net crossed with every process, so the totals scale
with the experiment grid, not with a deployment.

In [4]:
# ── T2: per-process fitting, per-case generation ────────────────────────────
_rows = []


def _row(stage, unit, total_s, n):
    """One stage row: its total seconds this run, its N units, and the ratio."""
    _scale = 1000.0 if unit.startswith('ms') else 1.0
    _rows.append({'Stage': stage, 'Cost': _scale * total_s / n if n else np.nan,
                  'Unit': unit, 'N': int(n), 'Total (s)': total_s})


_mine = rt[rt['stage'] == 'process_mining']
for alg, g in _mine.groupby('detail'):
    _row(f'Process discovery — {str(alg).capitalize()} miner', 's / process',
         g['seconds'].sum(), len(g))

_dur = rt[rt['stage'] == 'case_duration_training']
if not _dur.empty:
    _row('Case-duration model (budgeting)', 's / process',
         _dur['seconds'].sum(), len(_dur))

_MLP_LABEL = 'Duration ML models (ml_global + ml_local, one fit)'
_mlp = rt[rt['stage'] == 'duration_model_training']
if not _mlp.empty:
    _row(_MLP_LABEL, 's / process', _mlp['seconds'].sum(), len(_mlp))
else:
    # Fallback for runs made before this stage was instrumented: the log's own
    # timestamps, from the training banner to the 'feat_cols' line the fit prints
    # when it returns. ml_global and ml_local share one fit per process, so this is
    # the cost of both together.
    _t0, _spans = None, []
    for _l in (run / 'pipeline_execution.log').read_text(errors='replace').splitlines():
        _m = re.match(r'^(\d{4}-\d\d-\d\d \d\d:\d\d:\d\d,\d\d\d) - \w+ - (.*)$', _l)
        if not _m:
            continue
        _t, _msg = pd.Timestamp(_m.group(1).replace(',', '.')), _m.group(2).strip()
        if _msg.startswith('TRAINING ML+ MODELS (shared'):
            _t0 = _t
        elif _t0 is not None and _msg.startswith('feat_cols'):
            _spans.append((_t - _t0).total_seconds())
            _t0 = None
    if _spans:
        _row(_MLP_LABEL, 's / process', sum(_spans), len(_spans))
        print(f'Duration ML models: {sum(_spans):.1f} s over {len(_spans)} processes, '
              f'recovered from pipeline_execution.log — this run predates the '
              f'duration_model_training timer (added since).')
    else:
        print('WARNING: duration ML model training neither timed nor recoverable '
              'from the log. Row omitted.')

_TV_LABEL = {'baseline': 'sampled durations', 'ml_global': 'ML global durations',
             'ml_local': 'ML per-activity durations'}
_sim = rt[(rt['stage'] == 'simulation') & (rt['split'] == SPLIT)].copy()
_sim['tv']    = _sim['detail'].map(_time_variant)
_sim['cases'] = _sim['detail'].map(lambda m: cases.get((m, SPLIT), np.nan))
for tv, g in _sim.groupby('tv'):
    _row(f'Case generation — {_TV_LABEL.get(tv, tv)}', 'ms / case',
         g['seconds'].sum(), g['cases'].sum())

_pev = rt[(rt['stage'] == 'process_evaluation') & (rt['split'] == SPLIT)].copy()
_pev['cases'] = _pev['detail'].map(lambda m: cases.get((m, SPLIT), np.nan))
_row('Process-model scoring', 'ms / case', _pev['seconds'].sum(), _pev['cases'].sum())

_cc = rt[rt['stage'] == 'complete_curve_eval'].copy()
_cc['curve'] = _cc['detail'].astype(str).str.split(':').str[-1]
_cc['cases'] = (_cc['detail'].astype(str).str.split(':').str[0]
                   .map(lambda m: cases.get((m, SPLIT), np.nan)))
for appr, g in _cc.groupby('curve'):
    _name = METHOD_RENAME.get(APPROACH_EVAL_LABEL.get(appr, appr),
                              APPROACH_EVAL_LABEL.get(appr, appr))
    _row(f'Complete profile + scoring — {_name}', 'ms / case',
         g['seconds'].sum(), g['cases'].sum())

T2 = pd.DataFrame(_rows).set_index('Stage')[['Cost', 'Unit', 'N', 'Total (s)']]
display(T2.round(2))
print(f'Sum of the rows above: {T2["Total (s)"].sum()/60:.1f} min '
      f'({SPLIT} split; the fitting rows cover TRAIN)')

Duration ML models: 30.7 s over 6 processes, recovered from pipeline_execution.log — this run predates the duration_model_training timer (added since).


,Cost,Unit,N,Total (s)
Stage,,,,
Process discovery — Alpha miner,1.68,s / process,6,10.09
Process discovery — Heuristic miner,5.05,s / process,6,30.32
Process discovery — Inductive miner,5.07,s / process,6,30.39
Case-duration model (budgeting),0.51,s / process,6,3.04
"Duration ML models (ml_global + ml_local, one fit)",5.12,s / process,6,30.75
Case generation — sampled durations,1.13,ms / case,5544,6.25
Case generation — ML global durations,5.37,ms / case,5544,29.77
Case generation — ML per-activity durations,2.30,ms / case,5544,12.76
Process-model scoring,2.38,ms / case,16632,39.51


Sum of the rows above: 125.8 min (TEST split; the fitting rows cover TRAIN)


---
## 3 · LaTeX for the paper

`visuals/timing_results.tex` and `visuals/timing_process_stages.tex`.

In [5]:
# ── LaTeX: curve-generation cost ────────────────────────────────────────────
TIMING_MINIPAGE = '16cm'
TIMING_FIRSTCOL = '5.2cm'

# (T1 column, header, decimals, bold the minimum?). Bolding marks the cheapest, so
# it is only meaningful on a cost column — not on the model count, which is a
# property of the method's granularity.
TEX_COLS = [
    ('Curve models',             r'\makecell{Curve\\models}',               0, False),
    ('Training total (min)',     r'\makecell{Training\\total (CPU min)}',   2, True),
    ('Training per model (s)',   r'\makecell{Training per\\model (CPU s)}', 2, True),
    ('Inference per curve (ms)', r'\makecell{Inference per\\curve (ms)}',   1, True),
]

TIMING_NOTE = (
    'Cost of each curve-generation method on the run of '
    'Table~\\ref{{tab:individual_profile_realism}}. A curve model is one predictor per '
    '(sensor, activity, object); the sensor-median baseline fits one per sensor, so the '
    'per-model and per-curve columns are the comparable ones. Training is summed over the '
    '{workers} workers of the training pool, inference is wall clock over the {split} '
    'curves of a single process. Ratios, not absolute deployment figures. '
    '\\textbf{{Bold}} = cheapest per column.')


def _cells(tbl, cols):
    """Formatted cells, cheapest bolded. A positive cost that rounds to all zeros is
    printed as '<' the smallest representable value, never as a flat 0."""
    out = pd.DataFrame(index=tbl.index, columns=[c[0] for c in cols], dtype=object)
    for col, _hdr, dp, do_bold in cols:
        v    = pd.to_numeric(tbl[col], errors='coerce')
        best = v.min() if do_bold and v.notna().any() else None
        eps  = 10.0 ** -dp
        for idx in tbl.index:
            x = v.loc[idx]
            if pd.isna(x):
                s = '--'
            elif 0 < x < eps:
                s = r'$<$' + f'{eps:.{dp}f}'
            else:
                s = f'{x:,.{dp}f}'
            if best is not None and pd.notna(x) and abs(x - best) < 1e-12:
                s = r'\textbf{' + s + '}'
            out.loc[idx, col] = s
    return out


def t1_to_latex(tbl, caption, label, note):
    s    = _cells(tbl, TEX_COLS)
    hdr  = ' & '.join(['Method'] + [c[1] for c in TEX_COLS])
    body = '\n'.join(' & '.join([_tex_esc(idx)] + [s.loc[idx, c[0]] for c in TEX_COLS])
                     + r' \\' for idx in tbl.index)
    return '\n'.join([
        r'\begin{table*}[H]', r'\centering', '',
        rf'\begin{{minipage}}{{{TIMING_MINIPAGE}}}', r'\centering', '',
        r'\captionsetup{', r'    justification=centering,',
        r'    singlelinecheck=false,', r'    format=plain', r'}', '',
        rf'\caption{{{caption}}}', rf'\label{{{label}}}', '',
        r'\vspace{-0.5em}', '',
        rf'\begin{{tabular}}{{p{{{TIMING_FIRSTCOL}}}|' + '|'.join(['c'] * len(TEX_COLS)) + '}',
        r'\toprule', hdr + r' \\', r'\midrule', body, r'\bottomrule', r'\end{tabular}', '',
        r'\vspace{0.5em}', '',
        rf'\parbox{{{TIMING_MINIPAGE}}}{{%', r'\footnotesize', note, r'}', '',
        r'\end{minipage}', '', r'\end{table*}'])


tex_t1 = t1_to_latex(
    T1,
    caption='Computational cost of the curve-generation methods.',
    label='tab:timing_results',
    note=TIMING_NOTE.format(workers=POOL_WORKERS or 'pool', split=SPLIT.lower()))
print(tex_t1)

if SAVE_LATEX:
    Path('visuals').mkdir(exist_ok=True)
    _f = Path('visuals') / 'timing_results.tex'
    _f.write_text('% Curve-generation runtime — \\input{} this file.\n' + tex_t1 + '\n')
    print('\nSaved:', _f)

\begin{table*}[H]
\centering

\begin{minipage}{16cm}
\centering

\captionsetup{
    justification=centering,
    singlelinecheck=false,
    format=plain
}

\caption{Computational cost of the curve-generation methods.}
\label{tab:timing_results}

\vspace{-0.5em}

\begin{tabular}{p{5.2cm}|c|c|c|c}
\toprule
Method & \makecell{Curve\\models} & \makecell{Training\\total (CPU min)} & \makecell{Training per\\model (CPU s)} & \makecell{Inference per\\curve (ms)} \\
\midrule
ML Step DTW (proposed) & 390 & 246.65 & 37.95 & 22.6 \\
Median per Activity \& Sensor & 390 & \textbf{0.03} & \textbf{$<$0.01} & \textbf{6.6} \\
ML DTW (no steps) & 390 & 870.83 & 133.97 & 20.9 \\
ML (no DTW) & 390 & 869.53 & 133.77 & 16.6 \\
Sensor median & 79 & 0.65 & 0.49 & 6.8 \\
Seq2Seq DTW (aligned) & 390 & 164.83 & 25.36 & 41.5 \\
Seq2Seq (DTW-scored) & 390 & 678.74 & 104.42 & 36.3 \\
\bottomrule
\end{tabular}

\vspace{0.5em}

\parbox{16cm}{%
\footnotesize
Cost of each curve-generation method on the run of Table~\ref

In [6]:
# ── LaTeX: process-model stage cost ─────────────────────────────────────────
PROC_NOTE = (
    'Runtime of the process-model stages, same run. Fitting stages are per process, '
    'generation stages per generated case ({split} split, {n_proc} processes). Case '
    'generation is the simulator alone; complete profile adds the curve models over a '
    'whole case and the comparison against its real counterpart. Cost is single-process '
    'wall clock, a ratio rather than an absolute deployment figure; Total is what this '
    'run spent on the stage over its N units, so it scales with the experiment grid '
    '(every net crossed with every process).')


def t2_to_latex(tbl, caption, label, note):
    body = '\n'.join(' & '.join([_tex_esc(idx),
                                 f'{tbl.loc[idx, "Cost"]:,.2f}',
                                 _tex_esc(tbl.loc[idx, 'Unit']),
                                 f'{int(tbl.loc[idx, "N"]):,}',
                                 f'{tbl.loc[idx, "Total (s)"]:,.1f}']) + r' \\'
                     for idx in tbl.index)
    return '\n'.join([
        r'\begin{table}[H]', r'\centering', '',
        rf'\caption{{{caption}}}', rf'\label{{{label}}}', '',
        r'\vspace{-0.5em}', r'\footnotesize',
        r'\begin{tabular}{p{5.2cm}|r|l|r|r}', r'\toprule',
        r'Stage & Cost & Unit & N & Total (s) \\', r'\midrule',
        body, r'\bottomrule', r'\end{tabular}', '',
        r'\vspace{0.5em}', '',
        r'\parbox{\linewidth}{%', r'\footnotesize', note, r'}', '',
        r'\end{table}'])


tex_t2 = t2_to_latex(
    T2,
    caption='Computational cost of the process-model stages.',
    label='tab:timing_process_stages',
    note=PROC_NOTE.format(split=SPLIT.lower(), n_proc=N_PROCESSES))
print(tex_t2)

if SAVE_LATEX:
    _f = Path('visuals') / 'timing_process_stages.tex'
    _f.write_text('% Process-model stage runtime — \\input{} this file.\n' + tex_t2 + '\n')
    print('\nSaved:', _f)

\begin{table}[H]
\centering

\caption{Computational cost of the process-model stages.}
\label{tab:timing_process_stages}

\vspace{-0.5em}
\footnotesize
\begin{tabular}{p{5.2cm}|r|l|r|r}
\toprule
Stage & Cost & Unit & N & Total (s) \\
\midrule
Process discovery -- Alpha miner & 1.68 & s / process & 6 & 10.1 \\
Process discovery -- Heuristic miner & 5.05 & s / process & 6 & 30.3 \\
Process discovery -- Inductive miner & 5.07 & s / process & 6 & 30.4 \\
Case-duration model (budgeting) & 0.51 & s / process & 6 & 3.0 \\
Duration ML models (ml\_global + ml\_local, one fit) & 5.12 & s / process & 6 & 30.7 \\
Case generation -- sampled durations & 1.13 & ms / case & 5,544 & 6.2 \\
Case generation -- ML global durations & 5.37 & ms / case & 5,544 & 29.8 \\
Case generation -- ML per-activity durations & 2.30 & ms / case & 5,544 & 12.8 \\
Process-model scoring & 2.38 & ms / case & 16,632 & 39.5 \\
Complete profile + scoring -- Sensor median & 5.77 & ms / case & 16,632 & 95.9 \\
Complete profile +